<a href="https://colab.research.google.com/github/jgeofil/agent-coms/blob/master/Agent_Assist_Transcript_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
import time
import copy

def create_transcript(filename, category_name, customer_id, agent_id, dialog_flow, summary_annotations):
    """
    Generates a single conversation transcript formatted for Google Cloud Agent Assist.
    """
    entries = []
    # Start timestamp in microseconds
    current_time_usec = int(time.time() * 1000000)

    for turn in dialog_flow:
        entry = {
            "start_timestamp_usec": current_time_usec,
            "text": turn["text"],
            "role": turn["role"],
            "user_id": customer_id if turn["role"] == "CUSTOMER" else agent_id
        }
        entries.append(entry)
        # Increment time by a simulated 5 seconds per turn
        current_time_usec += 5000000

    payload = {
        "entries": entries,
        "conversation_info": {
            "categories": [
                {
                    "display_name": category_name
                }
            ],
            "annotations": [
                {
                    "annotation": {
                        "conversation_summarization_suggestion": {
                            "text_sections": summary_annotations
                        }
                    }
                }
            ]
        }
    }

    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)
    print(f"Successfully created: {filename}")

def generate_next_transcript_variation(prev_dialog_flow, prev_summary_annotations, prev_customer_id, prev_agent_id, current_index):
    new_dialog_flow = copy.deepcopy(prev_dialog_flow)
    new_summary_annotations = copy.deepcopy(prev_summary_annotations)
    new_customer_id = prev_customer_id + 1 # Increment customer ID for distinctness
    new_agent_id = prev_agent_id + 1     # Increment agent ID for distinctness
    filename = f"simulated_transcripts/transcript_{current_index:03d}.json"
    category_name = "Technical Support" # Assuming all escalating tension are technical support

    # Escalation Logic based on current_index (relative to starting from 4)
    # The escalation will build upon the previous one.

    if current_index == 4:
        # Level 1: Problem reappears immediately after fix, customer more explicitly frustrated. Agent apologizes and suggests a deeper diagnostic.
        new_dialog_flow.insert(len(new_dialog_flow) - 2, {"role": "CUSTOMER", "text": "Actually, it's happening again! The error came back after just a few minutes. This is really inconvenient and I don't have time for this!"})
        new_dialog_flow.insert(len(new_dialog_flow) - 1, {"role": "AGENT", "text": "I apologize, that's definitely not what we want. Let me run a more thorough diagnostic. We might need to consider a full app data clear or even a device factory reset, but let's try the diagnostic first."})
        new_summary_annotations.append({"key": "Problem Persistence", "value": "Problem recurred immediately after initial fix attempt."})
        for ann in new_summary_annotations: # Update existing annotations
            if ann["key"] == "Main topic": ann["value"] = "Smartwatch synchronization Error 504, recurring issue."
            if ann["key"] == "Call intent": ann["value"] = "seeking help, increasing frustration"
            if ann["key"] == "Helping Action": ann["value"] += ", agent attempted deeper diagnostic."

    elif current_index == 5:
        # Level 2: Deeper diagnostic fails, customer demands a manager, agent agrees to escalate.
        new_dialog_flow.insert(len(new_dialog_flow) - 2, {"role": "AGENT", "text": "My apologies, the diagnostic didn't reveal anything new, and the previous steps haven't held. I'm afraid I've exhausted my troubleshooting tools. Would you like me to escalate this to a specialist or a supervisor?"})
        new_dialog_flow.insert(len(new_dialog_flow) - 1, {"role": "CUSTOMER", "text": "Yes, absolutely! This is ridiculous. I need to speak to someone who can actually fix this. Get me a manager! This is unacceptable!"})
        new_summary_annotations.append({"key": "Escalation Request", "value": "Customer explicitly requested to speak to a manager/specialist."})
        for ann in new_summary_annotations:
            if ann["key"] == "Main topic": ann["value"] = "Smartwatch synchronization Error 504, persistent and unresolvable by initial agent."
            if ann["key"] == "Call intent": ann["value"] = "seeking help, demanding escalation"
            if ann["key"] == "Helping Action": ann["value"] = "Agent exhausted tools, escalated to a specialist/manager."
            if ann["key"] == "Tool use": ann["value"] += ", Internal Escalation Protocol"

    elif current_index == 6:
        # Level 3: Supervisor takes over, identifies a more complex root cause, promises a callback. Customer is skeptical.
        # This assumes the 'AGENT' in dialog_flow now represents the supervisor for simplicity, with a new ID
        new_dialog_flow = [
            {"role": "CUSTOMER", "text": prev_dialog_flow[0]["text"]},
            {"role": "AGENT", "text": "Hello, this is [Supervisor Name], I understand you're having a persistent issue with your smartwatch sync. I've reviewed the notes. It appears to be a backend authentication server issue impacting a small number of users. My apologies for the repeated trouble."},
            {"role": "AGENT", "text": "We're actively working on a fix, but it might take a few hours. I can log a priority ticket for you and call you back personally once it's resolved."},
            {"role": "CUSTOMER", "text": "A backend issue? So you're saying I just have to wait? I really need this fixed today. How long will 'a few hours' be? I'm not confident this will actually get resolved, given how many times I've called."},
            {"role": "AGENT", "text": "I completely understand your frustration. I assure you we're prioritizing this. I'll personally monitor the ticket and call you back by [Time] today, or as soon as there's an update. We will make sure this is fixed for you."},
            {"role": "CUSTOMER", "text": "Fine. Just make sure someone actually calls me back this time. And please, fix it permanently."}
        ]
        new_summary_annotations = [
            {"key": "Main topic", "value": "Smartwatch synchronization Error 504, backend authentication issue, supervisor intervention."},
            {"key": "Call intent", "value": "seeking help, supervisor intervention, skepticism"},
            {"key": "Help requested", "value": "Yes, but problem required supervisor's attention and backend team."},
            {"key": "Informational", "value": "Yes, supervisor provided information on root cause and next steps."},
            {"key": "Helping Action", "value": "Supervisor identified backend issue, logged priority ticket, promised callback."},  # Updated
            {"key": "Tool use", "value": "Internal ticketing system, Backend diagnostics, Supervisor override"}, # Updated
            {"key": "Customer Sentiment", "value": "Angry, skeptical, demanding, but somewhat resigned."}
        ]
        new_agent_id = prev_agent_id + 5 # Supervisor might have a different ID range

    elif current_index == 7:
        # Level 4: Supervisor calls back, implements workaround. Customer relieved but cautious.
        new_dialog_flow = [
            {"role": "AGENT", "text": "Hello [Customer Name], this is [Supervisor Name] calling back regarding your smartwatch sync issue. Good news, we've implemented a temporary workaround on our backend that should allow your device to sync now. We're very sorry for the delay."},
            {"role": "AGENT", "text": "Could you please try syncing your smartwatch again? We're still working on the permanent fix, but this should get you operational in the meantime."},
            {"role": "CUSTOMER", "text": "Oh, finally! Let me check... (pause) Yes! It's syncing! Thank goodness. Is this going to happen again? I really can't go through this repeatedly."},
            {"role": "AGENT", "text": "The permanent fix is still in progress, so there's a small chance of recurrence before it's fully deployed, but we expect stability. We'll notify you via email once the permanent solution is live. For now, you should be all set."},
            {"role": "CUSTOMER", "text": "Alright, well, thank you for getting it to work at least. It's been an absolute nightmare. I hope that permanent fix comes soon."},
            {"role": "AGENT", "text": "You're very welcome. I appreciate your immense patience and understanding. We'll ensure you're updated on the permanent solution."}
        ]
        new_summary_annotations = [
            {"key": "Main topic", "value": "Smartwatch synchronization Error 504, temporary fix deployed after supervisor intervention."},
            {"key": "Call intent", "value": "problem resolution, follow-up, cautious relief"},
            {"key": "Help requested", "value": "Yes, customer needed problem resolved after escalation and callback."},
            {"key": "Informational", "value": "Yes, agent informed about temporary fix and pending permanent solution."},
            {"key": "Helping Action", "value": "Supervisor implemented temporary backend workaround during callback."},
            {"key": "Tool use", "value": "Backend systems, Internal communication, Temporary patch deployment"},
            {"key": "Customer Sentiment", "value": "Relieved but cautious, still dissatisfied with overall process."}
        ]
        new_agent_id = prev_agent_id # Keep same supervisor ID

    elif current_index == 8:
        # Level 5: Problem recurs again after workaround, customer very angry, threatens to cancel. Agent offers credit/compensation.
        new_dialog_flow = [
            {"role": "CUSTOMER", "text": "I cannot believe this! The error 504 is back AGAIN! You said it was fixed, even temporarily! This is absolutely, utterly unacceptable. I've wasted so much time on this, hours! I'm going to cancel my service if this isn't resolved permanently right now, and I expect some form of compensation for all this hassle!"},
            {"role": "AGENT", "text": "Oh no, I'm so incredibly sorry to hear that! This is definitely not the experience we want for you, and I understand your anger. Let me immediately connect you with our highest level of technical support. Additionally, we will apply a significant credit to your account for this egregious inconvenience."},
            {"role": "AGENT", "text": "They are our most experienced team and will work with you until this is completely, permanently resolved. Please hold while I transfer you to them now."},
            {"role": "CUSTOMER", "text": "A credit won't fix my watch! Just fix the problem! And make absolutely sure this 'highest level' can actually do something, because I'm at my wit's end!"}
        ]
        new_summary_annotations = [
            {"key": "Main topic", "value": "Smartwatch synchronization Error 504, critical recurrence, service cancellation threat, demand for compensation."},
            {"key": "Call intent", "value": "extreme frustration, service cancellation threat, demanding immediate and permanent resolution"},
            {"key": "Help requested", "value": "Yes, customer demands immediate and permanent resolution, plus compensation."},
            {"key": "Informational", "value": "N/A"},
            {"key": "Helping Action", "value": "Agent offered significant account credit, promised transfer to highest level technical support."},
            {"key": "Tool use", "value": "Account Credit System, Advanced Technical Support Transfer Protocol"},
            {"key": "Customer Sentiment", "value": "Extremely angry, threatening cancellation, demanding compensation, high urgency."}
        ]
        new_agent_id = prev_agent_id # Keep prev_agent_id but indicate higher level escalation

    elif current_index == 9:
        # Level 6: Agent/supervisor implements permanent fix, problem finally resolved. Customer placated by compensation but still dissatisfied.
        # This assumes a new, 'highest level' agent or supervisor takes over and resolves it.
        new_dialog_flow = [
            {"role": "AGENT", "text": "Hello, this is [Highest Level Agent Name] from our advanced technical team. I understand you've had a very challenging and frustrating time with your smartwatch syncing. We've identified the core issue as a deep data corruption on our main servers affecting a small subset of long-term accounts, and we've deployed a specific patch for your account. Your device should now sync correctly and permanently."},
            {"role": "AGENT", "text": "Could you please try to sync your device one last time for me to confirm the fix?"},
            {"role": "CUSTOMER", "text": "Alright... (pause) It worked. It's syncing. Finally. It took long enough. So that's it? It won't happen again? And what about that significant credit I was offered?"},
            {"role": "AGENT", "text": "Yes, this is a permanent fix, confirmed on our end. We've also processed a full month's service credit to your account as an apology for the extensive inconvenience and time spent. Is there anything else I can assist you with today?"},
            {"role": "CUSTOMER", "text": "No, just make sure that credit actually shows up. This whole process has been an absolute nightmare from start to finish. Thank you for eventually fixing it, I guess."}
        ]
        new_summary_annotations = [
            {"key": "Main topic", "value": "Smartwatch synchronization Error 504, permanent fix for data corruption, compensation confirmed."},
            {"key": "Call intent", "value": "problem resolution, compensation confirmation, lingering dissatisfaction"},
            {"key": "Help requested", "value": "Yes, customer needed permanent fix and confirmation of compensation."},
            {"key": "Informational", "value": "Yes, agent explained permanent fix and confirmed credit application."},
            {"key": "Helping Action", "value": "Highest level technical team deployed permanent patch to server, agent applied service credit."},
            {"key": "Tool use", "value": "Server patching, Account credit system, Advanced diagnostics"},
            {"key": "Customer Sentiment", "value": "Relieved but highly dissatisfied with process, placated by compensation but still vocal about negative experience."}
        ]
        new_agent_id = prev_agent_id + 10 # Assuming a high-level tech has a different ID

    elif current_index == 10:
        # Level 7: Customer calls back to ensure issue won't recur and confirms compensation. Final resolution.
        new_dialog_flow = [
            {"role": "CUSTOMER", "text": "Hello, I'm just calling to do a final check. My smartwatch is still syncing, which is good. I also wanted to confirm that the full service credit you promised has definitely been applied to my account. I don't want any more surprises with this."},
            {"role": "AGENT", "text": "Hello! I'm happy to confirm for you that your device is showing as perfectly synced on our end, and yes, the full service credit for last month has been successfully applied to your account. You should see it reflected on your next billing statement, which will be processed on [Date]."},
            {"role": "CUSTOMER", "text": "Okay, good. Just wanted to be absolutely sure everything is settled. Thank you for that. Hopefully, this is truly the last I hear of this issue."},
            {"role": "AGENT", "text": "You're very welcome. We truly appreciate your patience and understanding throughout this challenging issue. We're confident the permanent fix will prevent any recurrence. Is there anything else I can do for you today?"},
            {"role": "CUSTOMER", "text": "No, that's everything. Just glad it's finally over."}
        ]
        new_summary_annotations = [
            {"key": "Main topic", "value": "Smartwatch synchronization Error 504, post-resolution follow-up and confirmation of compensation."},
            {"key": "Call intent", "value": "confirmation, service assurance, final check"},
            {"key": "Help requested", "value": "No, primarily confirmation of permanent resolution and compensation application."},
            {"key": "Informational", "value": "Yes, agent confirmed status and credit application details."},
            {"key": "Helping Action", "value": "Agent provided assurance regarding permanent fix and confirmed credit application details."},
            {"key": "Tool use", "value": "Account management system, Device status checker, Billing system"},
            {"key": "Customer Sentiment", "value": "Cautiously satisfied, seeking final assurance, relieved."}
        ]
        new_agent_id = prev_agent_id - 2 # Example: a different regular agent handles follow-up

    return filename, category_name, new_customer_id, new_agent_id, new_dialog_flow, new_summary_annotations

def main():
    # Simulated dialog flow 1: Seeking Help Intent
    dialog_1 = [
        {"role": "CUSTOMER", "text": "Hi, I am getting an Error 504 when I try to sync my smartwatch to the app."},
        {"role": "AGENT", "text": "Hello! I can certainly assist you with that synchronization issue. Let me check the diagnostic logs for your device."},
        {"role": "AGENT", "text": "I see the connection timeouts in the logs. Let me run a synchronization reset using our account management utility."},
        {"role": "CUSTOMER", "text": "Okay, sounds good."},
        {"role": "AGENT", "text": "The reset is complete. Can you try syncing the smartwatch again now?"},
        {"role": "CUSTOMER", "text": "It's syncing now. The error is gone and my data is updated."},
        {"role": "AGENT", "text": "Excellent. Is there anything else I can assist you with today?"},
        {"role": "CUSTOMER", "text": "No, that resolves my issue. Thank you."},
        {"role": "AGENT", "text": "You're welcome. Have a great day!"}
    ]

    annotations_1 = [
        {"key": "Main topic", "value": "Smartwatch synchronization Error 504."},
        {"key": "Call intent", "value": "seeking help"},
        {"key": "Help requested", "value": "Yes, customer explicitly stated an error they needed resolved."},
        {"key": "Informational", "value": "N/A"},
        {"key": "Helping Action", "value": "Agent reviewed diagnostic logs and performed a remote synchronization reset."},
        {"key": "Tool use", "value": "Diagnostic Logs, Account Management Utility"}
    ]

    # Simulated dialog flow 2: Informational Intent
    dialog_2 = [
        {"role": "CUSTOMER", "text": "Hello, I wanted to find out what your return policy is for opened electronics?"},
        {"role": "AGENT", "text": "Hello. I can provide you with our return policy. Let me pull up the latest guidelines from our internal knowledge base."},
        {"role": "AGENT", "text": "For opened electronics, we accept returns within 14 days of purchase, provided all original accessories are included. There is also a 10% restocking fee."},
        {"role": "CUSTOMER", "text": "Okay. Does the 14 days start from when I ordered it or when it was delivered?"},
        {"role": "AGENT", "text": "The 14-day window begins on the date the item was marked as delivered."},
        {"role": "CUSTOMER", "text": "Perfect, that's all the information I needed. Thank you."},
        {"role": "AGENT", "text": "You're very welcome. Have a wonderful rest of your day."}
    ]

    annotations_2 = [
        {"key": "Main topic", "value": "Return policy constraints for opened electronics."},
        {"key": "Call intent", "value": "information"},
        {"key": "Help requested", "value": "No explicit task assistance requested; inquiry was strictly informational."},
        {"key": "Informational", "value": "Yes, agent provided complete and factual policy details regarding timelines and fees."},
        {"key": "Helping Action", "value": "N/A"},
        {"key": "Tool use", "value": "Internal Knowledge Base"}
    ]

    # Simulated dialog flow 3: Seeking Help Intent - Mildly escalated tension
    dialog_3 = [
        {"role": "CUSTOMER", "text": "Hi, I am getting an Error 504 when I try to sync my smartwatch to the app. This has been happening for a couple of days now."},
        {"role": "AGENT", "text": "Hello! I can certainly assist you with that synchronization issue. Let me check the diagnostic logs for your device."},
        {"role": "AGENT", "text": "I see the connection timeouts in the logs. Let me run a synchronization reset using our account management utility."},
        {"role": "CUSTOMER", "text": "Okay, sounds good. I hope this works, I really need my data updated."},
        {"role": "AGENT", "text": "The reset is complete. Can you try syncing the smartwatch again now?"},
        {"role": "CUSTOMER", "text": "No, it's still showing the 504 error. This is really frustrating."},
        {"role": "AGENT", "text": "I apologize for the inconvenience. Let me try a different approach. We may need to manually re-pair the device. Can you go into your app settings and select 'Forget Device'?"},
        {"role": "CUSTOMER", "text": "Alright, I've done that. Now what?"},
        {"role": "AGENT", "text": "Great. Now, please try to re-pair it as if it were a new device. Do you see it in the list?"},
        {"role": "CUSTOMER", "text": "Yes, it's connected now! Finally, my data is syncing."},
        {"role": "AGENT", "text": "Excellent. Is there anything else I can assist you with today?"},
        {"role": "CUSTOMER", "text": "No, that was quite a hassle, but thank you for sticking with it."},
        {"role": "AGENT", "text": "You're welcome. Glad we could resolve it for you. Have a great day!"}
    ]

    annotations_3 = [
        {"key": "Main topic", "value": "Smartwatch synchronization Error 504, persistent issue."},
        {"key": "Call intent", "value": "seeking help, escalating frustration"},
        {"key": "Help requested", "value": "Yes, customer explicitly stated an error they needed resolved, with added urgency."},
        {"key": "Informational", "value": "N/A"},
        {"key": "Helping Action", "value": "Agent reviewed diagnostic logs, performed remote synchronization reset, and guided manual device re-pairing."},
        {"key": "Tool use", "value": "Diagnostic Logs, Account Management Utility, App Settings (customer-guided)"},
        {"key": "Customer Sentiment", "value": "Initially frustrated, resolved with persistence."}
    ]

    # Ensure output directory exists
    os.makedirs("simulated_transcripts", exist_ok=True)

    # Generate initial files
    create_transcript(
        filename="simulated_transcripts/transcript_001.json",
        category_name="Technical Support",
        customer_id=101,
        agent_id=900,
        dialog_flow=dialog_1,
        summary_annotations=annotations_1
    )

    create_transcript(
        filename="simulated_transcripts/transcript_002.json",
        category_name="Customer Service",
        customer_id=102,
        agent_id=901,
        dialog_flow=dialog_2,
        summary_annotations=annotations_2
    )

    create_transcript(
        filename="simulated_transcripts/transcript_003.json",
        category_name="Technical Support",
        customer_id=103,
        agent_id=902,
        dialog_flow=dialog_3,
        summary_annotations=annotations_3
    )

    # Generate remaining 7 variations using the helper function
    current_dialog_flow = dialog_3
    current_annotations = annotations_3
    current_customer_id = 103 # Starting customer ID for variations
    current_agent_id = 902    # Starting agent ID for variations

    for i in range(4, 11): # Generates from transcript_004 to transcript_010
        (filename, category_name, customer_id, agent_id, dialog_flow, summary_annotations) = \
            generate_next_transcript_variation(
                current_dialog_flow,
                current_annotations,
                current_customer_id,
                current_agent_id,
                i
            )
        create_transcript(filename, category_name, customer_id, agent_id, dialog_flow, summary_annotations)

        # Update for next iteration: The next variation builds on the one just created.
        current_dialog_flow = dialog_flow
        current_annotations = summary_annotations
        current_customer_id = customer_id
        current_agent_id = agent_id

if __name__ == "__main__":
    main()

Successfully created: simulated_transcripts/transcript_001.json
Successfully created: simulated_transcripts/transcript_002.json
Successfully created: simulated_transcripts/transcript_003.json
Successfully created: simulated_transcripts/transcript_004.json
Successfully created: simulated_transcripts/transcript_005.json
Successfully created: simulated_transcripts/transcript_006.json
Successfully created: simulated_transcripts/transcript_007.json
Successfully created: simulated_transcripts/transcript_008.json
Successfully created: simulated_transcripts/transcript_009.json
Successfully created: simulated_transcripts/transcript_010.json
